#  AegisHealth KBQA — Benchmark N3: Hybrid (Cypher + LightRAG)

Bậc 3 (đầy đủ) trong ablation 3 bậc:

```
N1 (LLM thuần)  →  N2 (+RAG vector / LightRAG naive)  →  N3 (+KG Hybrid Cypher+LightRAG)
```

Notebook này chạy **toàn bộ pipeline Hybrid** (`run_pipeline(question)`, router đầy đủ):
câu hỏi precise lookup → Cypher trực tiếp trên VietMedKG (Neo4j); câu hỏi semantic →
LightRAG `mix` mode.

## Kiến trúc
```
Câu hỏi (golden_test_v2.json)
        │
        
run_pipeline(q)  [in-process — không uvicorn, mode=None → router đầy đủ]
        │
   ┌────┴─────────────────────┐
                              
Cypher path                LightRAG mix
(Neo4j VietMedKG,           (Qdrant bge-m3 + LightRAG
 entity disambiguation)      internal graph trên Neo4j)
        │                           │
        └─────────────┬─────────────┘
                       
raw_hybrid.jsonl  ──  score_golden.score_run  ──  results_hybrid.json + report_hybrid.md
                       + routing distribution (cypher vs lightrag)
```

##  Checklist trước khi Run All
- [ ] **Accelerator** → GPU T4 x1
- [ ] **Internet** → bật
- [ ] **Add Data** → attach Kaggle Dataset `aegishealth-benchmark`
      (chứa `golden_test_v2.json` + `kg_entities.txt`)
- [ ] **Secrets**:
  - `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD` — Neo4j AuraDB (VietMedKG)
  - `QDRANT_URL`, `QDRANT_API_KEY` — Qdrant Cloud (bge-m3 1024d)
- [ ] Branch `refactor/review-architecture` đã được **push** lên GitHub

##  Lưu ý kỹ thuật
- **Embedding bắt buộc `bge-m3` (1024d)** — KHÔNG `nomic-embed-text` (768d).
- `DEFAULT_QUERY_MODE=mix` — LightRAG path dùng `mix` mode (Qdrant + LightRAG internal graph).
- Mỗi câu có thể gọi LLM 2-3 lần (intent extraction + synthesis) → chậm hơn N1/N2.

##  Ước tính: ~25-45 phút (100 câu)


In [ ]:
# --- Kiểm tra môi trường ---
import platform
import subprocess
import sys

print("=" * 60)
print(" Environment Check")
print("=" * 60)
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
        text=True,
    ).strip()
    print(f"GPU: {gpu_info}")
except Exception:
    print(" No GPU detected — bge-m3 + qwen2.5:3b sẽ chạy chậm hơn trên CPU")

with open("/proc/meminfo") as f:
    for line in f:
        if line.startswith("MemTotal"):
            ram_gb = int(line.split()[1]) / 1024 / 1024
            print(f"RAM: {ram_gb:.1f} GB")
            break

print("\n Environment check done")


In [ ]:
# --- Cấu hình + Kaggle Secrets ---
MODEL_NAME = "qwen2.5:3b"
EMBED_MODEL = "bge-m3"
EMBEDDING_DIM = 1024

OLLAMA_PORT = 11434
INFERENCE_TIMEOUT = 240  # giây / câu (khớp PIPELINE_TIMEOUT_SECONDS trong pipeline.py)

REPO_URL = "https://github.com/NgBaoAnn/kbqa.git"
REPO_BRANCH = "refactor/review-architecture"  # ← phải push trước khi chạy
REPO_DIR = "/kaggle/working/kbqa"

DATASET_DIR = "/kaggle/input/aegishealth-benchmark"
OUTPUT_DIR = "/kaggle/working"

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()


def get_secret(name, default=""):
    try:
        return secrets.get_secret(name)
    except Exception:
        print(f"   Secret '{name}' not found — using default")
        return default


NEO4J_URI = get_secret("NEO4J_URI")
NEO4J_USERNAME = get_secret("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD")
QDRANT_URL = get_secret("QDRANT_URL")
QDRANT_API_KEY = get_secret("QDRANT_API_KEY")

print(" Config loaded:")
print(f"  Model:     {MODEL_NAME}")
print(f"  Embedding: {EMBED_MODEL} ({EMBEDDING_DIM}d)")
print(f"  Neo4j URI: {NEO4J_URI[:40]}..." if NEO4J_URI else "  Neo4j URI:  NOT SET")
print(f"  Qdrant:    {QDRANT_URL[:40]}..." if QDRANT_URL else "  Qdrant:     NOT SET")


In [ ]:
# --- Clone repo + cài package (editable install) ---
import os
import subprocess
import sys

if os.path.exists(REPO_DIR):
    print(f" Repo đã tồn tại, đang cập nhật branch {REPO_BRANCH}...")
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_DIR, check=True)
else:
    print(f" Cloning repo {REPO_URL} (branch={REPO_BRANCH})...")
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, REPO_DIR],
        check=True,
    )

print(" Cài packages (lightrag-hku, neo4j, openai, qdrant-client...)")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "-q", "--no-warn-script-location"],
    cwd=REPO_DIR, capture_output=True, text=True,
)
if result.returncode != 0:
    print(" Lỗi cài package:")
    print(result.stderr[-3000:])
    raise RuntimeError("pip install -e . failed")
print(" Package cài xong")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qdrant-client"], check=True)
print(" qdrant-client cài xong")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from ai_engine.eval.score_golden import score_run

print(" score_golden imported OK")


In [ ]:
# --- Tìm golden_test_v2.json + kg_entities.txt ---
import json
from pathlib import Path


def find_file(name: str) -> Path:
    candidates = [
        Path(DATASET_DIR) / name,
        Path(REPO_DIR) / "data" / "benchmark" / name,
    ]
    for c in candidates:
        if c.exists():
            return c
    # Fallback: Kaggle có thể mount dataset ở đường dẫn lồng nhau
    # (vd. /kaggle/input/datasets/<owner>/<slug>/...) — quét đệ quy /kaggle/input.
    for c in Path("/kaggle/input").rglob(name):
        return c
    raise FileNotFoundError(
        f"{name} không tìm thấy trong {DATASET_DIR}, {REPO_DIR}/data/benchmark, "
        f"hoặc bất kỳ đâu dưới /kaggle/input — kiểm tra Kaggle Dataset 'aegishealth-benchmark' đã attach chưa."
    )


GOLDEN_PATH = find_file("golden_test_v2.json")
KG_PATH = find_file("kg_entities.txt")

with open(GOLDEN_PATH, encoding="utf-8") as f:
    golden = json.load(f)

print(f" Golden:      {GOLDEN_PATH}  ({len(golden)} câu)")
print(f" KG entities: {KG_PATH}")


In [ ]:
# --- Cài và khởi động Ollama, pull LLM + embedding ---
import os
import subprocess
import time

import requests

OLLAMA_URL = f"http://localhost:{OLLAMA_PORT}"
OLLAMA_MODELS_DIR = "/kaggle/working/ollama_models"


def find_ollama():
    for p in ("/usr/local/bin/ollama", "/usr/bin/ollama", "/bin/ollama"):
        if os.path.exists(p):
            return p
    try:
        return subprocess.check_output(["which", "ollama"], text=True).strip()
    except Exception:
        return None


ollama_bin = find_ollama()

if ollama_bin:
    print(f" Ollama đã có sẵn tại: {ollama_bin}")
else:
    print("  Đang cài Ollama...")
    install_result = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True,
    )
    print(install_result.stdout[-1000:])
    ollama_bin = find_ollama()
    if not ollama_bin:
        raise RuntimeError(f"Ollama install failed (code {install_result.returncode})")
    print(f" Ollama đã cài thành công tại: {ollama_bin}")

os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
print(" Khởi động Ollama server...")
ollama_env = os.environ.copy()
ollama_env["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR
ollama_env["OLLAMA_KEEP_ALIVE"] = "60m"

ollama_log = "/kaggle/working/ollama.log"
with open(ollama_log, "w") as log:
    ollama_proc = subprocess.Popen([ollama_bin, "serve"], env=ollama_env, stdout=log, stderr=log)

print(" Chờ Ollama server...")
for i in range(40):
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=2)
        if r.status_code == 200:
            print(f" Ollama server ready (sau {i + 1}s)")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama server timeout sau 40s")

for model in (MODEL_NAME, EMBED_MODEL):
    print(f"  Pulling model: {model} (có thể mất vài phút)")
    t0 = time.time()
    result = subprocess.run([ollama_bin, "pull", model], text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ollama pull {model} failed")
    print(f" '{model}' sẵn sàng ({time.time() - t0:.0f}s)")


In [ ]:
# --- Cấu hình env (in-process, KHÔNG uvicorn) + import run_pipeline ---
# QUAN TRỌNG: set os.environ TRƯỚC khi import bất kỳ module nào trong
# ai_engine/backend, vì các hằng số config được đọc tại import-time.
import os
import sys

REPO_BACKEND_DIR = os.path.join(REPO_DIR, "backend")
for p in (REPO_DIR, REPO_BACKEND_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(REPO_DIR)

os.environ.update({
    # Neo4j AuraDB — VietMedKG (Cypher path) + LightRAG internal graph
    "NEO4J_URI": NEO4J_URI,
    "NEO4J_USERNAME": NEO4J_USERNAME,
    "NEO4J_PASSWORD": NEO4J_PASSWORD,
    # LLM (Ollama local)
    "LLM_BASE_URL": f"http://localhost:{OLLAMA_PORT}/v1",
    "LLM_MODEL_NAME": MODEL_NAME,
    "LLM_TIMEOUT_SECONDS": "90",
    # Embedding — bge-m3 1024d (KHÔNG nomic-embed-text)
    "EMBEDDING_MODEL": EMBED_MODEL,
    "EMBEDDING_DIM": str(EMBEDDING_DIM),
    "EMBEDDING_BASE_URL": f"http://localhost:{OLLAMA_PORT}/v1",
    # LightRAG — mix mode, Qdrant vector storage + Neo4j graph storage
    "LIGHTRAG_WORKING_DIR": os.path.join(REPO_DIR, "lightrag_data"),
    "LIGHTRAG_KG_STORAGE": "Neo4JStorage",
    "LIGHTRAG_VECTOR_STORAGE": "QdrantVectorDBStorage",
    "LIGHTRAG_DOC_STORAGE": "JsonKVStorage",
    "DEFAULT_QUERY_MODE": "mix",
    "FORCE_LIGHTRAG_NAIVE_MODE": "false",
    # Router đầy đủ — Cypher path BẬT
    "DISABLE_CYPHER_PATH": "false",
    # Qdrant Cloud
    "QDRANT_URL": QDRANT_URL,
    "QDRANT_API_KEY": QDRANT_API_KEY,
    "LOG_LEVEL": "WARNING",
})

from backend.app.services.pipeline import run_pipeline
from ai_engine import config as ai_config

print(" run_pipeline imported (in-process)")
print(f"  EMBEDDING_MODEL:          {ai_config.EMBEDDING_MODEL} ({ai_config.EMBEDDING_DIM}d)")
print(f"  LIGHTRAG_KG_STORAGE:      {ai_config.LIGHTRAG_KG_STORAGE}")
print(f"  LIGHTRAG_VECTOR_STORAGE:  {ai_config.LIGHTRAG_VECTOR_STORAGE}")
print(f"  DEFAULT_QUERY_MODE:       {ai_config.DEFAULT_QUERY_MODE}")
print(f"  DISABLE_CYPHER_PATH:      {os.environ['DISABLE_CYPHER_PATH']}")

warnings = ai_config.validate_config()
for w in warnings:
    print(f"   {w}")


In [ ]:
# --- Smoke test — xác nhận có câu engine=cypher_direct ---
test_q = golden[0]["question"]
print(f" Smoke test: {test_q}")

result = await run_pipeline(test_q)
meta = result.get("metadata", {})
print(f"status:     {result.get('status')}")
print(f"engine:     {meta.get('engine')}")
print(f"query_mode: {meta.get('query_mode')}")
print(f"answer:\n{str(result.get('answer', ''))[:500]}")

if meta.get("engine") != "cypher_direct":
    print(
        "\n Câu smoke test KHÔNG route vào Cypher path. "
        "Kiểm tra kết nối Neo4j / entity disambiguation trước khi chạy full benchmark."
    )
else:
    print("\n Cypher path hoạt động.")


In [ ]:
# --- Chạy inference cho toàn bộ golden set (router đầy đủ) ---
import json
import time

RAW_PATH = os.path.join(OUTPUT_DIR, "raw_hybrid.jsonl")

t_start = time.time()
with open(RAW_PATH, "w", encoding="utf-8") as f:
    for i, item in enumerate(golden):
        q = item["question"]
        t0 = time.time()
        try:
            result = await run_pipeline(q)
            elapsed_ms = round((time.time() - t0) * 1000, 1)
            meta = result.get("metadata", {})
            row = {
                **item,
                "system_answer": result.get("answer", ""),
                "engine": meta.get("engine"),
                "query_mode": meta.get("query_mode"),
                "error_code": meta.get("error_code"),
                "elapsed_ms": elapsed_ms,
            }
        except Exception as e:
            elapsed_ms = round((time.time() - t0) * 1000, 1)
            row = {
                **item,
                "system_answer": "",
                "engine": None,
                "query_mode": None,
                "error_code": "EXCEPTION",
                "elapsed_ms": elapsed_ms,
            }
            print(f"   Exception: {e}")

        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()

        print(
            f"[{i + 1}/{len(golden)}] {elapsed_ms:7.0f}ms "
            f"engine={row['engine']!s:13} mode={row['query_mode']!s:24} "
            f"err={row['error_code']!s:5} {q[:40]}"
        )

print(f"\n Done in {time.time() - t_start:.0f}s → {RAW_PATH}")


In [ ]:
# --- Chấm điểm bằng score_golden ---
from collections import Counter

report = score_run(RAW_PATH, GOLDEN_PATH, KG_PATH, use_bertscore=False)

RESULTS_PATH = os.path.join(OUTPUT_DIR, "results_hybrid.json")
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

s = report["summary"]
print(f"{'=' * 55}")
print("  SCORE REPORT — N3 Hybrid (Cypher + LightRAG)")
print(f"{'=' * 55}")
print(f"  Total:        {s['total']}  (errors: {s['error_count']}, timeouts: {s['timeout_count']})")
print(f"  Precision:    {s['precision']:.3f}")
print(f"  Recall:       {s['recall']:.3f}")
print(f"  F1:           {s['f1']:.3f}")
print(f"  Fabrication:  {s['fabrication_rate']:.3f}  ")
print(f"  Off-answer:   {s['off_answer_rate']:.3f}")

engines = Counter(r.get("engine") for r in report["rows"])
print(f"\n  Engine distribution: {dict(engines)}")
print(f"\n Results saved → {RESULTS_PATH}")


In [ ]:
# --- Sinh report_hybrid.md + biểu đồ routing distribution ---
import matplotlib.pyplot as plt


def fmt_row(name, m):
    if not m:
        return f"| {name} | - | - | - | - | - | - | - |"
    return (
        f"| {name} | {m['n']} | {m['precision']:.3f} | {m['recall']:.3f} | "
        f"{m['f1']:.3f} | {m['fabrication_rate']:.3f} | {m['em']:.3f} | {m['hits1']:.3f} |"
    )


lines = []
lines.append(f"# Báo cáo Benchmark — N3: Hybrid (Cypher + LightRAG mix, {MODEL_NAME} + {EMBED_MODEL})\n")
lines.append(
    f"**Tổng số câu:** {s['total']}  •  **Lỗi:** {s['error_count']}  •  "
    f"**Timeout:** {s['timeout_count']}\n"
)
lines.append("## Tổng quan\n")
lines.append(f"- Precision: **{s['precision']:.3f}**")
lines.append(f"- Recall: **{s['recall']:.3f}**")
lines.append(f"- F1: **{s['f1']:.3f}**")
lines.append(f"- Fabrication rate: **{s['fabrication_rate']:.3f}** ")
lines.append(f"- Off-answer rate: **{s['off_answer_rate']:.3f}**")
lines.append(f"- Engine distribution: `{dict(engines)}`\n")

lines.append("## Theo regime\n")
lines.append("| regime | n | P | R | F1 | Fab | EM | Hits@1 |")
lines.append("|---|---|---|---|---|---|---|---|")
for regime, m in report["by_regime"].items():
    lines.append(fmt_row(regime, m))
lines.append("")

lines.append("## Theo complexity (hop)\n")
lines.append("| complexity | n | P | R | F1 | Fab | EM | Hits@1 |")
lines.append("|---|---|---|---|---|---|---|---|")
for hop, m in report["by_complexity"].items():
    lines.append(fmt_row(hop, m))
lines.append("")

lines.append("## Theo question_type\n")
lines.append("| question_type | n | P | R | F1 | Fab | EM | Hits@1 |")
lines.append("|---|---|---|---|---|---|---|---|")
for qt, m in report["by_type"].items():
    lines.append(fmt_row(qt, m))
lines.append("")

REPORT_PATH = os.path.join(OUTPUT_DIR, "report_hybrid.md")
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f" Report saved → {REPORT_PATH}")

from IPython.display import Markdown, display

display(Markdown("\n".join(lines)))

# Routing distribution chart
fig, ax = plt.subplots(figsize=(5, 4))
labels = list(engines.keys())
values = [engines[k] for k in labels]
ax.bar([str(label) for label in labels], values, color=["#4C72B0", "#DD8452", "#999999"])
ax.set_title("N3 — Routing distribution (engine)")
ax.set_ylabel("# câu hỏi")
for i, v in enumerate(values):
    ax.text(i, v + 0.5, str(v), ha="center")
plt.tight_layout()
chart_path = os.path.join(OUTPUT_DIR, "routing_distribution.png")
plt.savefig(chart_path, dpi=120)
plt.show()
print(f" Chart saved → {chart_path}")


In [ ]:
# --- Dọn dẹp ---
try:
    ollama_proc.terminate()
    print(" Ollama stopped")
except Exception as e:
    print(f" {e}")

print("\n Output files:")
for fn in (
    "raw_hybrid.jsonl", "results_hybrid.json", "report_hybrid.md", "routing_distribution.png",
):
    p = os.path.join(OUTPUT_DIR, fn)
    if os.path.exists(p):
        print(f"  {p} ({os.path.getsize(p):,} bytes)")
